# ✍️ Option 4 — Content Generator
## AI Engineer Ready · Cohort 1 · Assignment 1

---

## 📋 README — Read This First!

### 🎯 What This Assignment Is About
You will build a **multi-step AI content pipeline** that:
1. Takes a **topic** as input
2. Generates a structured **outline** (using an LLM)
3. Expands that outline into **full blog-style content** (using an LLM)
4. Produces a short **summary** of the full content (using an LLM)

The core idea is **chained LLM calls** — each step feeds into the next. This is the foundation of real-world AI pipelines.

---

### 🗂️ Files in This Project
| File | Purpose |
|---|---|
| `Content_Generator.ipynb` | This notebook — your main workspace |
| `main.py` | The same logic, packaged as a CLI script |
| `requirements.txt` | Python dependencies (`openai`, `langchain`) |
| `README.md` | Short project description |

---

### 🛠️ Tools & Libraries You Will Use
- **LangChain** — framework for chaining LLM calls
- **PromptTemplate** — structured, reusable prompts
- **OpenAI API** (or any compatible LLM)
- **Python f-strings / string formatting**

---

### 📊 How You Will Be Evaluated
| Criteria | What the Evaluator Looks For | Marks |
|---|---|---|
| **Setup & Installation** | Environment runs without errors | 10% |
| **Outline Generation** | LLM returns a meaningful, structured outline | 20% |
| **Content Expansion** | Each outline point is expanded into coherent paragraphs | 25% |
| **Summarization** | Summary is concise and captures the content | 20% |
| **Prompt Quality** | Prompts are clear, specific, and use PromptTemplate | 15% |
| **Code Clarity** | Clean code, comments, no leftover debug prints | 10% |

---

### ✅ Definition of Done
- [ ] All cells run top-to-bottom without errors
- [ ] Output is visible in the notebook (not just in `main.py`)
- [ ] Each of the 3 functions (`generate_outline`, `generate_content`, `summarize`) uses a real LLM call via LangChain
- [ ] `PromptTemplate` is used (not raw f-strings inside the LLM call)
- [ ] You can change the `TOPIC` variable in Section 6 and the whole pipeline re-runs correctly

---

> 💡 **Tip:** Work section by section. Each section builds on the previous one. Don't skip ahead!

---
# Section 1 — Environment Setup & Imports

## 📖 What to Do Here
Before writing any logic, you need to install the required libraries and import them.

### Step-by-step:
1. Run the `pip install` cell to install dependencies from `requirements.txt`
2. Import all the modules you will need throughout this notebook
3. Set your **OpenAI API key** (use an environment variable — never hardcode it!)

### 🔑 Getting Your API Key
- Sign up at https://platform.openai.com
- Go to **API Keys** and create a new secret key
- Store it safely — you'll set it as an environment variable below

### ⚠️ Evaluation Note
If this section throws an import error, the rest of the notebook cannot run. Make sure this section is 100% green before moving on.

In [ ]:
# Install dependencies
# Run this cell once. After installation, you can comment it out.
%pip install -r requirements.txt -q

In [ ]:
# TODO: Import required libraries
# Hint: You will need:
#   - os  (for reading environment variables)
#   - from langchain_core.prompts import PromptTemplate
#   - from langchain_openai import ChatOpenAI  (or from langchain.llms import OpenAI)
#   - from langchain.chains import LLMChain  (optional — you can also call the LLM directly)

import os

# TODO: Add your imports below
# from langchain_core.prompts import PromptTemplate
# from langchain_openai import ChatOpenAI

print("✅ Imports successful")

In [ ]:
# TODO: Set your OpenAI API key
# NEVER paste your key as a plain string in the notebook.
# Use one of these two safe approaches:

# APPROACH A — Set it in your terminal before launching Jupyter:
#   Windows:  set OPENAI_API_KEY=sk-...
#   Mac/Linux: export OPENAI_API_KEY=sk-...
# Then this line will pick it up automatically:
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

# APPROACH B — Use getpass (hides your key from display):
# import getpass
# OPENAI_API_KEY = getpass.getpass("Enter your OpenAI API Key: ")
# os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

if not OPENAI_API_KEY:
    print("⚠️  WARNING: OPENAI_API_KEY is not set. LLM calls will fail.")
else:
    print(f"✅ API key loaded (ends with: ...{OPENAI_API_KEY[-4:]})")

---
# Section 2 — Initialise the LLM

## 📖 What to Do Here
Create a single shared LLM object that all three pipeline steps will reuse.

### Concepts to Understand
- **`ChatOpenAI`** is a LangChain wrapper around OpenAI's chat models (e.g. `gpt-4o-mini`)
- **`temperature`** controls creativity: `0.0` = deterministic, `1.0` = very creative. Start with `0.7`.
- **`model_name`** — use `"gpt-4o-mini"` (cheap & fast for testing)

### 🔍 What the Evaluator Checks
- A single `llm` object is created here and reused below (not re-created in every function)
- Temperature is explicitly set

In [ ]:
# TODO: Initialise the LLM
# Replace the None below with a real ChatOpenAI instance.
#
# Example:
#   llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.7)
#
# Docs: https://python.langchain.com/docs/integrations/chat/openai

llm = None  # TODO: replace with ChatOpenAI(...)

if llm is not None:
    print(f"✅ LLM initialised: {llm.model_name}")
else:
    print("⚠️  LLM is not yet initialised.")

---
# Section 3 — Step 1: Generate an Outline

## 📖 What to Do Here
Implement the `generate_outline(topic)` function. It should ask the LLM to produce a structured outline for a blog post on the given topic.

### How It Works
```
topic (string)  ──►  PromptTemplate  ──►  LLM  ──►  outline (list of strings)
```

### Step-by-step Instructions
1. **Create a `PromptTemplate`** with a variable `{topic}`. The prompt should instruct the LLM to return a numbered outline with 4-6 sections.
2. **Format the prompt** by calling `prompt.format(topic=topic)`
3. **Call the LLM** with the formatted prompt
4. **Parse the response** — split by newlines, strip empty lines, return a Python list

### 💡 Suggested Prompt (you can improve it!)
```
You are a professional content strategist.
Create a detailed blog post outline for the topic: "{topic}".
Return ONLY a numbered list of 5 section headings, one per line.
Do not include any explanation.
```

### 🔍 What the Evaluator Checks
- `PromptTemplate` is used (not a raw f-string)
- Function returns a Python `list`, not a raw string
- Output has at least 3 meaningful headings for any topic

In [ ]:
# TODO: Define the outline prompt template
# Hint: Use PromptTemplate(input_variables=["topic"], template="...your prompt...")

outline_prompt = None  # TODO: replace with PromptTemplate(...)

In [ ]:
def generate_outline(topic: str) -> list:
    """
    Uses an LLM to generate a structured outline for a blog post on `topic`.

    Args:
        topic (str): The subject of the blog post.

    Returns:
        list: A list of section heading strings.

    TODO Steps:
        1. Format outline_prompt with the given topic
        2. Invoke the LLM with the formatted prompt
        3. Extract the text from the response  ->  response.content  (for ChatOpenAI)
        4. Split the text by newlines
        5. Strip whitespace from each line and remove empty lines
        6. Return the resulting list
    """
    # TODO: implement this function
    # Step 1: formatted_prompt = outline_prompt.format(topic=topic)
    # Step 2: response = llm.invoke(formatted_prompt)
    # Step 3: text = response.content
    # Step 4-6: parse and return
    pass


# Quick test
test_outline = generate_outline("Getting started with LangChain")
print("Generated Outline:")
for i, point in enumerate(test_outline or [], 1):
    print(f"  {i}. {point}")

---
# Section 4 — Step 2: Generate Full Content

## 📖 What to Do Here
Implement `generate_content(outline)`. It takes the list of headings from Step 1 and asks the LLM to write a full paragraph for each section.

### How It Works
```
outline (list)  ──►  PromptTemplate  ──►  LLM  ──►  full_content (string)
```

### Step-by-step Instructions
1. **Join the outline list** into a single string (one heading per line) so it can be injected into the prompt
2. **Create a `PromptTemplate`** with variable `{outline}`. The prompt should ask the LLM to write 2-3 paragraphs per section.
3. **Format & invoke** the LLM
4. Return the full content as a single string

### 💡 Suggested Prompt
```
You are an expert blog writer.
Using the following outline, write a complete blog post.
For each section heading, write 2-3 informative paragraphs.
Use markdown formatting (## for headings).

Outline:
{outline}
```

### 🔍 What the Evaluator Checks
- Input is the outline list from Step 1 (not hardcoded)
- Each outline heading appears in the output as a section
- Content is coherent and meaningful (not placeholder text)
- Returns a string, not a list

In [ ]:
# TODO: Define the content prompt template

content_prompt = None  # TODO: replace with PromptTemplate(...)

In [ ]:
def generate_content(outline: list) -> str:
    """
    Expands an outline into a full blog post using an LLM.

    Args:
        outline (list): List of section headings from generate_outline().

    Returns:
        str: The full blog post content as a markdown-formatted string.

    TODO Steps:
        1. Join the outline list into a single string:  outline_str = "\n".join(outline)
        2. Format content_prompt with outline_str
        3. Invoke the LLM
        4. Return response.content
    """
    # TODO: implement this function
    pass


# Quick test
test_content = generate_content(test_outline or ["Introduction", "Key Concepts", "Conclusion"])
print("Generated Content (first 500 chars):")
print((test_content or "")[:500], "...")

---
# Section 5 — Step 3: Summarize the Content

## 📖 What to Do Here
Implement `summarize(content)`. It takes the full blog post and produces a concise 3-5 sentence summary.

### How It Works
```
full_content (string)  ──►  PromptTemplate  ──►  LLM  ──►  summary (string)
```

### Step-by-step Instructions
1. **Create a `PromptTemplate`** with variable `{content}`.
2. **Be careful with long content** — you can truncate the input to the first 3000 characters if needed to stay within token limits.
3. **Format & invoke** the LLM
4. Return the summary string

### 💡 Suggested Prompt
```
You are a content editor. Summarize the following blog post in 3-5 sentences.
Be concise and capture the key points.

Blog Post:
{content}

Summary:
```

### 🔍 What the Evaluator Checks
- Summary is 3-5 sentences (not a one-liner, not a full rewrite)
- Summary accurately reflects the content (not generic filler)
- Input comes from Step 2's output (not hardcoded)

In [ ]:
# TODO: Define the summarization prompt template

summary_prompt = None  # TODO: replace with PromptTemplate(...)

In [ ]:
def summarize(content: str) -> str:
    """
    Produces a 3-5 sentence summary of a blog post using an LLM.

    Args:
        content (str): The full blog post text from generate_content().

    Returns:
        str: A short summary paragraph.

    TODO Steps:
        1. Optionally truncate content to avoid token limits:
               content_trimmed = content[:3000]
        2. Format summary_prompt with content_trimmed
        3. Invoke the LLM
        4. Return response.content
    """
    # TODO: implement this function
    pass


# Quick test
test_summary = summarize(test_content or "Sample content for testing.")
print("Summary:")
print(test_summary)

---
# Section 6 — 🚀 Run the Full Pipeline

## 📖 What to Do Here
Now that all three functions are implemented, wire them together into a single end-to-end pipeline.

### Instructions
1. Set your `TOPIC` below (change it to anything you like!)
2. Call `generate_outline` → `generate_content` → `summarize` in sequence
3. Print each result with a clear label

### 🔍 What the Evaluator Checks
- Changing `TOPIC` and re-running produces different, topic-appropriate output
- All three outputs are printed and visible in the notebook
- No hardcoded strings — everything flows from the `TOPIC` variable

In [ ]:
# Configuration — change this to test your pipeline!
TOPIC = "Getting started with LangChain"   # TODO: try different topics!

print(f"🔄 Running Content Generator Pipeline for topic: '{TOPIC}'")
print("=" * 60)

# Step 1: Generate Outline
print("\n📝 STEP 1 — Generating Outline...")
outline = None  # TODO: replace with generate_outline(TOPIC)

if outline:
    print("Outline:")
    for i, point in enumerate(outline, 1):
        print(f"  {i}. {point}")
else:
    print("⚠️  Outline not generated yet.")

# Step 2: Generate Content
print("\n📄 STEP 2 — Generating Full Content...")
content = None  # TODO: replace with generate_content(outline)

if content:
    print("Content (first 800 chars):")
    print(content[:800], "...")
else:
    print("⚠️  Content not generated yet.")

# Step 3: Summarize
print("\n🔍 STEP 3 — Summarizing...")
summary = None  # TODO: replace with summarize(content)

if summary:
    print("Summary:")
    print(summary)
else:
    print("⚠️  Summary not generated yet.")

print("\n" + "=" * 60)
print("✅ Pipeline complete!")

---
# Section 7 — Self-Assessment Checklist

Before submitting, go through this checklist. Be honest with yourself!

Run the cell below and update the `checklist` dictionary — set each item to `True` when you've completed it.

In [ ]:
checklist = {
    "All cells run without errors":                      False,
    "API key loaded via environment variable":           False,
    "LLM object initialised with explicit temperature":  False,
    "PromptTemplate used in all 3 functions":            False,
    "generate_outline returns a list":                   False,
    "generate_content returns a string with sections":   False,
    "summarize returns a 3-5 sentence summary":          False,
    "Pipeline runs end-to-end from TOPIC variable":      False,
    "Changing TOPIC produces correct new output":        False,
    "Code is clean and commented":                       False,
}

print("📋 Self-Assessment Results")
print("=" * 50)
total = len(checklist)
done = sum(checklist.values())
for item, status in checklist.items():
    icon = "✅" if status else "❌"
    print(f"  {icon}  {item}")
print("=" * 50)
print(f"  Score: {done}/{total} items complete ({int(done/total*100)}%)")
if done == total:
    print("  🎉 You're ready to submit!")
else:
    print(f"  ⏳ {total - done} item(s) still to go.")